# AE 370 Project 1 — Method section studies

Numerical evidence for Section 2 of the report. Every time integrator below is written from scratch; `fsolve` is used only for the algebraic solve inside the implicit methods.

| Notebook part | Report section |
|---|---|
| 1. Jacobian & eigenvalues | 2.2 |
| 2. Truncation error & convergence | 2.3 |
| 3. Absolute stability & stiffness | 2.4, 2.5 |
| 4. Comparison of methods | 2.6 |

`feed_model_upgrade.py` must be in the same folder, or its contents pasted into the first code cell in place of the import. Run the cells top to bottom; the full notebook takes about 1–2 minutes.

In [1]:
import os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd                       # !pip install pandas if not installed
from scipy.optimize import fsolve
from feed_model_upgrade import *          # model: f, jacobian, info, ... (or paste the module here)

p  = make_params()                        # nominal case: P_set = P_u0 = 360 psig
u0 = initial_state(p)
T_END = 27.6                              # fixed horizon just before burnout (t_f ~ 27.9 s); = 69 x 0.4 s
TOL   = 1e-2                              # accuracy target: 1% on mdot_f
np.seterr(all="ignore"); warnings.filterwarnings("ignore")   # unstable runs overflow on purpose
FIG_DIR = "figures"; os.makedirs(FIG_DIR, exist_ok=True)     # report figures are saved here as PDF

ModuleNotFoundError: No module named 'feed_model_upgrade'

## Time integrators

Lecture 10–13 notation: `u` is the state and `y1..y4` are RK stages. `advance()` takes one step $u_k \to u_{k+1}$ for any method. `march()` repeats it, starts multi-step methods with RK4 (Lecture 11), and stops if the solution blows up.

In [ ]:
class Counted:
    "Wraps a function and counts its calls (cost = evaluations of f and J)."
    def __init__(self, fun): self.fun, self.n = fun, 0
    def __call__(self, u):
        self.n += 1
        return self.fun(u)

AB   = {"AB2": [-1/2, 3/2], "AB4": [-9/24, 37/24, -59/24, 55/24]}   # beta, oldest first (Lecture 11)
HIST = {"AB2": 2, "AB4": 4, "BDF2": 2}                                # past values each multi-step method needs
SCALE = np.abs(u0)                                                    # states differ by 1e4 in size -> scale the solve

def rk4_step(rhs, u, dt):
    y1 = rhs(u); y2 = rhs(u + dt/2*y1); y3 = rhs(u + dt/2*y2); y4 = rhs(u + dt*y3)
    return u + dt/6*(y1 + 2*y2 + 2*y3 + y4)

def solve_implicit(G, dG, guess):
    "Solve G(z) = 0 with fsolve, using the analytic Jacobian dG, in scaled variables x = z / SCALE."
    s = SCALE
    x = fsolve(lambda x: G(x*s)/s, guess/s, fprime=lambda x: dG(x*s)*s[None, :]/s[:, None], xtol=1e-12)
    return x*s

def advance(method, Uh, Fh, dt, rhs, jac):
    "One step. Uh = [..., u_{k-1}, u_k]; Fh = [..., f(u_{k-1}), f(u_k)] (Adams-Bashforth only)."
    u, I = Uh[-1], np.eye(5)
    if method == "FE":
        return u + dt*rhs(u)
    if method == "Heun":
        y1 = rhs(u); y2 = rhs(u + dt*y1)
        return u + dt/2*(y1 + y2)
    if method == "RK4":
        return rk4_step(rhs, u, dt)
    if method in AB:                                    # u_{k+1} = u_k + dt * sum(beta_j f_j)
        b = AB[method]
        return u + dt*sum(bj*fj for bj, fj in zip(b, Fh[-len(b):]))
    if method == "BE":                                  # z - u_k - dt f(z) = 0
        return solve_implicit(lambda z: z - u - dt*rhs(z), lambda z: I - dt*jac(z), u + dt*rhs(u))
    if method == "Trap":                                # z - u_k - dt/2 (f(u_k) + f(z)) = 0
        fu = rhs(u)
        return solve_implicit(lambda z: z - u - dt/2*(fu + rhs(z)), lambda z: I - dt/2*jac(z), u + dt*fu)
    if method == "BDF2":                                # 3z - 4u_k + u_{k-1} - 2 dt f(z) = 0
        return solve_implicit(lambda z: 3*z - 4*u + Uh[-2] - 2*dt*rhs(z), lambda z: 3*I - 2*dt*jac(z), u + dt*rhs(u))
    raise ValueError(method)

def march(method, dt, n):
    "n steps of size dt from u0. Returns U (rows = u_k; nan after a blow-up) and call counts."
    rhs, jac = Counted(lambda u: f(u, p)), Counted(lambda u: jacobian(u, p))
    U = np.full((n + 1, 5), np.nan); U[0] = u0
    Fh = [rhs(u0)] if method in AB else []
    try:
        for k in range(n):
            m = method if k + 1 >= HIST.get(method, 1) else "RK4"     # RK4 starter
            un = advance(m, U[:k + 1], Fh, dt, rhs, jac)
            if not np.all(np.isfinite(un)) or np.any(un[:4] <= 0):    # blew up / unphysical
                break
            U[k + 1] = un
            if method in AB:
                Fh.append(rhs(un))
    except (ValueError, ArithmeticError):
        pass
    return U, {"f": rhs.n, "J": jac.n}

## Reference solution and error metric

The reference is RK4 with $\Delta t_{ref} = 0.4/2^{10}$ s. Every $\Delta t$ in the sweeps is $0.4/2^j$, so the grids line up exactly. The error metric is Eq. (target) of the report: the maximum relative error of $\dot m_f$ over the burn.

In [ ]:
DT_REF = 0.4/2**10
N_REF  = int(round(T_END/DT_REF))
T_REF  = np.arange(N_REF + 1)*DT_REF
t0 = time.perf_counter()
U_REF, _ = march("RK4", DT_REF, N_REF)
MF_REF = np.array([mdot_f(u, p) for u in U_REF])
print(f"reference: {N_REF} RK4 steps in {time.perf_counter() - t0:.1f} s")

def err_mdotf(U, dt):
    "e_mdotf = max_k |mdot_f(u_k) - mdot_f(u(t_k))| / mdot_f(u(t_k));  nan if the run blew up."
    if np.isnan(U).any():
        return np.nan
    ref = MF_REF[::int(round(dt/DT_REF))][:len(U)]
    m = np.array([mdot_f(u, p) for u in U])
    return np.max(np.abs(m - ref)/ref)

# reference accuracy: RK4 at 2*dt_ref has ~2^4 = 16x the error of the reference itself
U2, _ = march("RK4", 2*DT_REF, N_REF//2)
print(f"estimated reference error in mdot_f: {err_mdotf(U2, 2*DT_REF)/15:.1e}")
print(f"regimes visited after t = 0: {set(info(u, p)['regime'] for u in U_REF[1:])}")

## 1. Jacobian & eigenvalues  (report Sec. 2.2)

First, check the analytic Jacobian, Eqs. (13)–(17), against central differences. Then compute the eigenvalues at states along the reference solution.

In [ ]:
# --- analytic J vs central differences along the burn ---
chk = np.linspace(1, N_REF, 25).astype(int)
rel = [np.max(np.abs(jacobian(U_REF[i], p) - jacobian_fd(U_REF[i], p)))/np.max(np.abs(jacobian_fd(U_REF[i], p)))
       for i in chk]
print(f"analytic vs central-difference Jacobian: max relative difference = {max(rel):.1e}")

# --- eigenvalues every 0.0125 s, sorted ascending: lam1 (fastest decay) ... lam5 ---
IDX_E = np.r_[1, np.arange(32, N_REF + 1, 32)]
T_E   = T_REF[IDX_E]
EIG   = np.array([np.linalg.eigvals(jacobian(U_REF[i], p)) for i in IDX_E])
print(f"largest |imaginary part| of any eigenvalue: {np.abs(EIG.imag).max():.1e}  (all real)")
LAM   = np.sort(EIG.real, axis=1)
i_max = np.argmax(-LAM[:, 0])
LAM_MAX, T_LAM_MAX = -LAM[i_max, 0], T_E[i_max]
print(f"|lambda|_max = {LAM_MAX:.3f} 1/s at t* = {T_LAM_MAX:.3f} s")

# --- Table 1 of the report ---
T_STAR = [T_REF[1], T_LAM_MAX, 1, 5, 10, 20, T_END]
def lam_at(t):
    return np.sort(np.linalg.eigvals(jacobian(U_REF[int(round(t/DT_REF))], p)).real)
tab1 = pd.DataFrame([lam_at(t) for t in T_STAR], columns=[f"lam{j}" for j in range(1, 6)])
tab1.insert(0, "t* [s]", np.round(T_STAR, 3))
display(tab1.style.format({"t* [s]": "{:.3f}", **{f"lam{j}": "{:.3e}" for j in range(1, 6)}}))

In [ ]:
# --- what each mode is (report Sec. 2.2 bullets) ---
c = np.array([1, 0, 1, 0, 0])                      # N2 mass: m_p + m_u
rows = []
for t in T_STAR[1:]:
    u = U_REF[int(round(t/DT_REF))]; J = jacobian(u, p); lam = np.sort(np.linalg.eigvals(J).real)
    blk = np.sort(np.roots([1, -J[1, 1], -J[0, 1]*J[1, 0]]).real)    # 2x2 tank block (m_p, T_p)
    rows.append([t, lam[1]/(info(u, p)["mdot_p"]/u[2]),               # lam2 / (mdot_p/m_u)
                 lam[2], blk[0], lam[4], blk[1],                       # lam3, lam5 vs block roots
                 np.abs(c @ J).max()/np.abs(J).max()])                 # c^T J = 0 -> lam4 = 0
modes = pd.DataFrame(rows, columns=["t* [s]", "lam2/(mdot_p/m_u)", "lam3", "lam3 (2x2)",
                                    "lam5", "lam5 (2x2)", "|c^T J|/|J|"])
display(modes.style.format({"t* [s]": "{:.2f}", "lam2/(mdot_p/m_u)": "{:.3f}", "|c^T J|/|J|": "{:.0e}",
                            **{k: "{:.3e}" for k in ["lam3", "lam3 (2x2)", "lam5", "lam5 (2x2)"]}}))

fig, ax = plt.subplots(figsize=(7, 4))
for j, lab in [(0, r"$|\lambda_1|$ ullage pressure"), (1, r"$|\lambda_2|$ ullage mixing"),
               (2, r"$|\lambda_3|$ tank cooling"), (4, r"$\lambda_5$ tank feedback ($>0$)")]:
    ax.semilogy(T_E, np.abs(LAM[:, j]), label=lab)
ax.plot(T_LAM_MAX, LAM_MAX, "ko"); ax.annotate(rf"  $|\lambda|_{{max}}$ = {LAM_MAX:.2f} s$^{{-1}}$", (T_LAM_MAX, LAM_MAX))
ax.set(xlabel="t [s]", ylabel=r"$|\lambda|$ [1/s]", title=r"Eigenvalues of $J$ along the burn ($\lambda_4 = 0$ not shown)")
ax.set_ylim(1e-5, 30); ax.legend(fontsize=8, loc="center right", bbox_to_anchor=(1, 0.62)); ax.grid(True, which="both", alpha=0.3)
fig.savefig(f"{FIG_DIR}/fig_eigenvalues.pdf", bbox_inches="tight"); plt.show()

## 2. Truncation error & convergence  (report Sec. 2.3)

- **(a) Truncation error** $\tau_k$ (Lecture 11): one step from *exact* data (the reference), divided by $\Delta t$, at $t_k = 2$ s. It should fall as $\Delta t^p$, even where the method is unstable.
- **(b) Global error** $e_{\dot m_f}$ over the whole burn. It falls as $\Delta t^p$ only where the method is also stable; dashed vertical lines mark each explicit method's predicted wall, $\Delta t_{stab} = w^*/|\lambda|_{max}$.

In [ ]:
#            order p, implicit?, w* (Lecture 12; None = whole negative real axis)
METHODS = {"FE": (1, False, 2.0), "Heun": (2, False, 2.0), "RK4": (4, False, 2.785),
           "AB2": (2, False, 1.0), "AB4": (4, False, 0.3),
           "BE": (1, True, None), "Trap": (2, True, None), "BDF2": (2, True, None)}
COL = dict(zip(METHODS, plt.cm.tab10.colors))
STY = lambda m: "--" if METHODS[m][1] else "-"

# --- (a) truncation error at t_k = 2 s ---
def tau(method, dt, tk=2.0):
    "tau_k = (u(t_k+1) - u~_k+1)/dt, u~ = one step from exact history. Max relative component [1/s]."
    s, k = int(round(dt/DT_REF)), int(round(tk/DT_REF))
    ex = lambda j: U_REF[k + j*s]                               # exact u(t_k + j dt)
    rhs, jac = (lambda u: f(u, p)), (lambda u: jacobian(u, p))
    n = HIST.get(method, 1)
    Uh = [ex(j) for j in range(1 - n, 1)]
    un = advance(method, Uh, [rhs(v) for v in Uh], dt, rhs, jac)
    return np.max(np.abs((ex(1) - un)/ex(0)))/dt

DTS_TAU = 0.4/2**np.arange(6)                                   # 0.4 ... 0.0125 s
TAU = {m: [tau(m, dt) for dt in DTS_TAU] for m in METHODS}

# --- (b) global error over the burn: the step-size sweep used for everything below ---
DTS = 0.4/2**np.arange(9)                                       # 0.4 ... 1.6e-3 s
RUNS = {}                                                       # (method, dt) -> err, f/J counts, wall time
t0 = time.perf_counter()
for m, (pm, impl, _) in METHODS.items():
    for dt in (DTS[:8] if impl else DTS):                        # implicit runs are slow; skip the finest dt
        tw = time.perf_counter()
        U, cnt = march(m, dt, int(round(T_END/dt)))
        RUNS[m, dt] = dict(err=err_mdotf(U, dt), time=time.perf_counter() - tw, **cnt)
print(f"sweep: {len(RUNS)} runs in {time.perf_counter() - t0:.0f} s")
dts_of = lambda m: np.array(sorted(dt for (mm, dt) in RUNS if mm == m))
err_of = lambda m: np.array([RUNS[m, dt]["err"] for dt in dts_of(m)])

# --- largest step meeting the 1% target, found with real runs (used in Secs. 3 and 4) ---
def run_s(m, s):
    "Run m at dt = s*dt_ref over N_REF//s steps (ends within one dt of T_END)."
    dt = s*DT_REF
    tw = time.perf_counter()
    U, cnt = march(m, dt, N_REF//s)
    return dict(dt=dt, err=err_mdotf(U, dt), time=time.perf_counter() - tw, **cnt)

def dt_1pct(m):
    "Bracket from the sweep (last passing, first failing dt), then bisect on the integer s."
    d, e = dts_of(m), err_of(m)
    i = 0
    while i + 1 < len(d) and e[i + 1] <= TOL:                    # nan (blow-up) compares False -> stops
        i += 1
    best = dict(dt=d[i], **RUNS[m, d[i]])
    if i + 1 == len(d):
        return best
    lo, hi = int(round(d[i]/DT_REF)), int(round(d[i + 1]/DT_REF))
    while hi - lo > 1:
        mid = (lo + hi)//2
        r = run_s(m, mid)
        lo, hi, best = (mid, hi, r) if r["err"] <= TOL else (lo, mid, best)
    return best

RUN_1 = {m: dt_1pct(m) for m in METHODS}
DT_1 = {m: RUN_1[m]["dt"] for m in METHODS}
print("dt meeting 1% [s]:", {m: round(v, 4) for m, v in DT_1.items()})

In [ ]:
from matplotlib.ticker import NullFormatter
def guides(ax, x0, x1, y0, orders=(1, 2, 4)):
    "Fan of slope guides from (x0, y0) to x1, labelled at the right end."
    for q in orders:
        x = np.array([x0, x1]); y = y0*(x/x0)**q
        ax.loglog(x, y, color="k", lw=0.9, ls="--", alpha=0.6)
        ax.text(x1*1.05, y[1], f"slope {q}", fontsize=7, va="center")

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for m, (pm, impl, ws) in METHODS.items():
    axs[0].loglog(DTS_TAU, TAU[m], STY(m), marker="o", ms=4, color=COL[m], label=f"{m} (p={pm})")
    axs[1].loglog(dts_of(m), err_of(m), STY(m), marker="o", ms=4, color=COL[m], label=f"{m} (p={pm})")
    if ws:                                                      # predicted stability wall (explicit only)
        axs[1].axvline(ws/LAM_MAX, color=COL[m], ls=":", lw=1.2)
guides(axs[0], 0.0125, 0.04, 1e-13)
guides(axs[1], 1.6e-3, 6e-3, 3e-12)
axs[0].set(title=r"(a) truncation error $\tau_k$ at $t_k$ = 2 s", ylabel=r"$\max_i |\tau_{k,i}/u_i(t_k)|$ [1/s]", ylim=(5e-14, 1e-3))
axs[1].set(title=r"(b) global error over the burn", ylabel=r"$e_{\dot m_f}$ (max relative)", ylim=(1e-12, 10))
axs[1].axhline(TOL, color="k", lw=1); axs[1].text(1.7e-3, TOL*1.4, "1% target", fontsize=8)
axs[1].text(1.7e-3, 3e-3, r"dotted verticals: predicted $\Delta t_{stab}$" "\nmissing points: run blew up", fontsize=7, va="top")
for ax in axs:
    ax.set_xlabel(r"$\Delta t$ [s]"); ax.grid(True, which="both", alpha=0.3)
    ax.xaxis.set_minor_formatter(NullFormatter())
axs[0].legend(fontsize=8, ncol=2, loc="lower right")
plt.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_convergence.pdf", bbox_inches="tight"); plt.show()

# observed order: least-squares slope over the 3 smallest dt with error above the reference floor (1e-11)
def observed_p(dts, errs):
    ok = np.isfinite(errs) & (errs > 1e-11)
    d, e = dts[ok][:3], errs[ok][:3]
    return np.polyfit(np.log(d), np.log(e), 1)[0]
OBS_P = {m: observed_p(dts_of(m), err_of(m)) for m in METHODS}

# accuracy-only step from the asymptotic fit e ~ C dt^p (ignores stability) -- for the stiffness test
def dt_acc_fit(m):
    pm = METHODS[m][0]; d, e = dts_of(m), err_of(m)
    ok = np.isfinite(e) & (e > 1e-11)
    C = np.median(e[ok][:3]/d[ok][:3]**pm)
    return (TOL/C)**(1/pm)
DT_ACC = {m: dt_acc_fit(m) for m in METHODS}
OBS_TAU = {m: np.polyfit(np.log(DTS_TAU[-3:]), np.log(TAU[m][-3:]), 1)[0] for m in METHODS}
display(pd.DataFrame({"p (theory)": {m: v[0] for m, v in METHODS.items()},
                      "slope of tau_k": OBS_TAU, "slope of e_mdotf": OBS_P}).T.style.format("{:.2f}"))

## 3. Absolute stability & stiffness  (report Sec. 2.4, 2.5)

The model problem $\dot u = \lambda u$ with $w = \Delta t\lambda$ (Lecture 12):
- one-step methods give $u_{k+1} = R(w)\,u_k$, stable where $|R(w)| < 1$;
- multi-step methods are stable where every root $\zeta$ of the characteristic polynomial has $|\zeta| < 1$.

All our eigenvalues are real, so the number that matters is $w^*$, where the region crosses the negative real axis.

In [ ]:
R_ONE = {"FE":   lambda w: 1 + w,
         "Heun": lambda w: 1 + w + w**2/2,
         "RK4":  lambda w: 1 + w + w**2/2 + w**3/6 + w**4/24,
         "BE":   lambda w: 1/(1 - w),
         "Trap": lambda w: (1 + w/2)/(1 - w/2)}
CHAR  = {"AB2":  lambda w: [1, -(1 + 1.5*w), 0.5*w],                              # zeta^2 - zeta = w(3/2 zeta - 1/2)
         "AB4":  lambda w: [1, -1 - 55/24*w, 59/24*w, -37/24*w, 9/24*w],
         "BDF2": lambda w: [3 - 2*w, -4, 1]}                                     # 3zeta^2 - 4zeta + 1 = 2w zeta^2

def amp(m, w):
    "Growth factor per step: |R(w)| (one-step) or the largest |zeta| (multi-step)."
    return abs(R_ONE[m](w)) if m in R_ONE else np.max(np.abs(np.roots(CHAR[m](w))))

def w_star(m, wmax=50.0):
    "First w < 0 (moving left from 0) where the method stops being stable; None if stable to -wmax."
    ws = -np.linspace(1e-3, wmax, 5001)
    bad = [w for w in ws if amp(m, w) >= 1]
    if not bad:
        return None
    lo, hi = bad[0] + (ws[0] - ws[1]), bad[0]                     # bracket, then bisect
    for _ in range(50):
        mid = (lo + hi)/2
        lo, hi = (mid, hi) if amp(m, mid) < 1 else (lo, mid)
    return -(lo + hi)/2

W_STAR = {m: w_star(m) for m in METHODS}
display(pd.DataFrame({"w* computed": {m: W_STAR[m] if W_STAR[m] else np.inf for m in METHODS},
                      "w* Lecture 12": {m: v[2] if v[2] else np.inf for m, v in METHODS.items()}}).T
        .style.format("{:.4f}"))

In [ ]:
EXPL = [m for m in METHODS if W_STAR[m]]
fig, axs = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.1, 1]})

# (a) regions of absolute stability + RK4's operating points w_j = dt * lambda_j(t)
x, y = np.meshgrid(np.linspace(-3.2, 0.8, 241), np.linspace(-3.2, 3.2, 241))
W = x + 1j*y
for m in EXPL:
    A = np.vectorize(lambda w: amp(m, w))(W)
    axs[0].contour(x, y, A, levels=[1], colors=[COL[m]], linewidths=1.6)
    axs[0].plot([], [], color=COL[m], label=f"{m}: $w^*$ = {W_STAR[m]:.3f}")
w_rk4 = DT_1["RK4"]*LAM[:, :3].ravel()                           # decaying modes, all t
axs[0].plot(w_rk4[::10], 0*w_rk4[::10], "k.", ms=3, label=rf"$w_j = \Delta t\,\lambda_j(t)$, RK4 at $\Delta t$ = {DT_1['RK4']:.3f} s")
axs[0].axhline(0, color="gray", lw=0.5); axs[0].axvline(0, color="gray", lw=0.5)
axs[0].set(xlabel="Re(w)", ylabel="Im(w)", title="(a) Regions of absolute stability (inside each curve)", aspect="equal")
axs[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
axs[0].text(-3.1, -3.0, "BE, Trap, BDF2: whole left half-plane is stable", fontsize=8)

# (b) each explicit method at its own 1% step: is the fastest mode inside [-w*, 0]?
for i, m in enumerate(EXPL):
    axs[1].plot([-W_STAR[m], 0], [i, i], color=COL[m], lw=8, alpha=0.35, solid_capstyle="butt")
    w_worst = -DT_1[m]*LAM_MAX
    axs[1].plot(w_worst, i, "o", color=COL[m], ms=8)
    axs[1].text(w_worst, i + 0.22, rf"$\Delta t_{{1\%}}$ = {DT_1[m]:.3g} s", fontsize=8, ha="center")
axs[1].set_yticks(range(len(EXPL))); axs[1].set_yticklabels(EXPL)
axs[1].set(xlabel=r"$w$ on the negative real axis", xlim=(-3.2, 0.2), ylim=(-0.5, len(EXPL) - 0.3),
           title="(b) Fastest mode at each method's 1% step")
axs[1].text(-3.15, -0.42, r"bar: stable interval $[-w^*, 0]$     dot: $w = -\Delta t_{1\%}\,|\lambda|_{max}$", fontsize=8)
axs[1].grid(True, axis="x", alpha=0.3)
plt.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_stability.pdf", bbox_inches="tight"); plt.show()

**Stiffness.** $R_s = |\lambda_1|/|\lambda_3|$ (decaying modes) says the problem is stiff. What matters, though, is whether stability forces a smaller step than accuracy does. Compare $\Delta t_{stab}(t) = w^*/|\lambda_1(t)|$ with the step each method needs for 1%.

In [ ]:
RS = np.abs(LAM[:, 0]/LAM[:, 2])
after = T_E >= T_LAM_MAX                                       # after the regulator has opened
print(f"stiffness ratio R_s = |lam1|/|lam3|: {RS[after].max():.0f} at t = {T_LAM_MAX:.2f} s, "
      f"{RS[-1]:.0f} at t = {T_E[-1]:.1f} s")

fig, ax = plt.subplots(figsize=(8, 4.5))
for m in EXPL:
    lab = "FE = Heun" if m == "FE" else m
    if m != "Heun":                                            # same w* as FE -> same curve
        ax.semilogy(T_E, W_STAR[m]/np.abs(LAM[:, 0]), color=COL[m], label=rf"{lab}: $\Delta t_{{stab}}(t)$")
    ax.axhline(DT_1[m], color=COL[m], ls="--", lw=1)
ax.plot([], [], "k--", lw=1, label=r"dashed: $\Delta t_{1\%}$ of each method")
ax.set(xlabel="t [s]", ylabel=r"$\Delta t$ [s]", title=r"Stability limit (solid) vs. step needed for 1% on $\dot m_f$ (dashed)")
ax.legend(fontsize=8, ncol=2, loc="lower right"); ax.grid(True, which="both", alpha=0.3)
fig.savefig(f"{FIG_DIR}/fig_stiffness.pdf", bbox_inches="tight"); plt.show()

# step counts over the burn: stability alone vs. accuracy alone (Sec. 2.5, Eq. Nstab)
stiff = pd.DataFrame({m: {"dt_stab [s]": W_STAR[m]/LAM_MAX, "dt_acc (accuracy only) [s]": DT_ACC[m],
                          "N_stab": T_END*LAM_MAX/W_STAR[m], "N_acc": T_END/DT_ACC[m]} for m in EXPL}).T
stiff["step set by"] = np.where(stiff["N_acc"] >= stiff["N_stab"], "accuracy", "stability")
display(stiff.style.format({"dt_stab [s]": "{:.3f}", "dt_acc (accuracy only) [s]": "{:.3f}",
                            "N_stab": "{:.0f}", "N_acc": "{:.0f}"}))

AB4's measured 1% step sits above its predicted $\Delta t_{stab}$. $|\lambda_1|$ is near its maximum for only a few seconds, so a $w$ just outside the region amplifies errors only for a short time; Lecture 12 likewise found measured walls slightly above the predicted ones. That step lies on the steep, instability-driven part of AB4's error curve, not the $\Delta t^4$ part (next cell): a slightly larger step, or a configuration with a larger $|\lambda|_{max}$, fails the target.

In [ ]:
# AB4 across its stability wall: smooth dt^4 error up to w ~ -0.3, then a steep rise
for s in [80, 96, 104, 112, 116, 120, 124, 128]:
    r = run_s("AB4", s)
    print(f"dt = {r['dt']:.4f} s   w = {-r['dt']*LAM_MAX:.3f}   e_mdotf = {r['err']:.2e}")
out = T_E[DT_1["AB4"]*np.abs(LAM[:, 0]) > W_STAR["AB4"]]          # times where w_1 is outside AB4's region
print(f"at dt_1% = {DT_1['AB4']:.4f} s, w_1 lies outside AB4's region only for {out.min():.2f} s < t < {out.max():.2f} s")

## 4. Comparison of methods  (report Sec. 2.6, Table 2)

Every column comes from an actual run at the largest step meeting the 1% target (found by bisection in Sec. 2), over one burn of $T$ = 27.6 s. "Step set by" compares $\Delta t_{stab}$ with the accuracy-only step from the fit $e \approx C\Delta t^p$.

In [ ]:
rows = {}
for m, (pm, impl, _) in METHODS.items():
    r = RUN_1[m]
    dt_stab = W_STAR[m]/LAM_MAX if W_STAR[m] else np.inf
    steps = N_REF//int(round(r["dt"]/DT_REF))
    rows[m] = {"p": pm, "observed p": OBS_P[m], "implicit": "yes" if impl else "no",
               "w*": W_STAR[m] if W_STAR[m] else np.inf, "dt_stab [s]": dt_stab, "dt_1% [s]": r["dt"],
               "e_mdotf": r["err"], "step set by": "stability" if dt_stab < DT_ACC[m] else "accuracy",
               "steps": steps, "f-evals": r["f"], "f-evals/step": r["f"]/steps, "J-evals": r["J"],
               "wall time [s]": r["time"]}
table2 = pd.DataFrame(rows).T
display(table2.style.format({"observed p": "{:.2f}", "w*": "{:.3f}", "dt_stab [s]": "{:.3f}", "dt_1% [s]": "{:.4f}",
                             "e_mdotf": "{:.1e}", "steps": "{:.0f}", "f-evals": "{:.0f}", "f-evals/step": "{:.1f}",
                             "J-evals": "{:.0f}", "wall time [s]": "{:.3f}"}))

# work-precision: error vs. cost (f-evals + J-evals) for every run in the sweep
fig, ax = plt.subplots(figsize=(7, 4.5))
for m in METHODS:
    d = dts_of(m)
    cost = [RUNS[m, x]["f"] + RUNS[m, x]["J"] for x in d]
    ax.loglog(cost, err_of(m), STY(m), marker="o", ms=4, color=COL[m], label=m)
ax.axhline(TOL, color="k", lw=1)
ax.set(xlabel="evaluations of f and J per run", ylabel=r"$e_{\dot m_f}$", title="Work-precision (lower left is better)")
ax.legend(fontsize=8, ncol=2); ax.grid(True, which="both", alpha=0.3)
fig.savefig(f"{FIG_DIR}/fig_work_precision.pdf", bbox_inches="tight"); plt.show()

## 5. Algorithm: RK4 to burnout  (report Sec. 2.7)

Fixed-step RK4 until the step that would take $m_f$ below zero. That last step is shortened to $h \le \Delta t$, with $h$ found by Newton's method on $g(h) = m_f$ after an RK4 step of length $h$. Since $dm_f/dt = -\dot m_f$, $g'(h) \approx -\dot m_f$.

In [ ]:
def rk4_burn(dt, u_init=u0, tol_mf=1e-10):
    "RK4 from u_init until m_f = 0. Returns times, states and the burnout time t_f."
    rhs = lambda u: f(u, p)
    T, U = [0.0], [u_init.copy()]
    while True:
        un = rk4_step(rhs, U[-1], dt)
        if un[4] > 0:                                   # full step, fuel left
            T.append(T[-1] + dt); U.append(un); continue
        # last step: Newton on h so that m_f(u_k -> h) = 0, starting from the linear estimate
        uk = U[-1]
        h = uk[4]/mdot_f(uk, p)
        for _ in range(20):
            uh = rk4_step(rhs, uk, h)
            if abs(uh[4]) < tol_mf*u_init[4]:
                break
            h += uh[4]/mdot_f(uh, p)                    # g(h)/(-g'(h)),  g' = dm_f/dt = -mdot_f
        T.append(T[-1] + h); U.append(uh)
        return np.array(T), np.array(U), T[-1]

T_ref_b, U_ref_b, TF_REF = rk4_burn(DT_REF)
print(f"burnout (dt_ref): t_f = {TF_REF:.6f} s, m_f(t_f) = {U_ref_b[-1, 4]:.1e} kg")
rows = []
for dt in [0.4, 0.2, 0.1, 0.05, 0.025]:
    _, Ub, tf = rk4_burn(dt)
    rows.append([dt, tf, abs(tf - TF_REF), Ub[-1, 4]])
display(pd.DataFrame(rows, columns=["dt [s]", "t_f [s]", "|t_f - t_f,ref| [s]", "m_f(t_f) [kg]"])
        .style.format({"dt [s]": "{:.3f}", "t_f [s]": "{:.6f}", "|t_f - t_f,ref| [s]": "{:.1e}", "m_f(t_f) [kg]": "{:.1e}"}))

# operating envelope of the nominal burn (quoted in the report)
S = [info(u, p) for u in U_ref_b]
print(f"regimes after t = 0: {set(s['regime'] for s in S[1:])}")
print(f"P_p: {min(s['P_p'] for s in S)/PSI - 14.7:.0f} - {max(s['P_p'] for s in S)/PSI - 14.7:.0f} psig")
print(f"P_u - P_e: {min(s['P_u'] - p['P_C'] for s in S)/PSI:.1f} - {max(s['P_u'] - p['P_C'] for s in S)/PSI:.1f} psi")
print(f"mdot_f: {S[0]['mdot_f']:.3f} -> {S[-1]['mdot_f']:.3f} kg/s;  amplification P_u/(2(P_u-P_e)): "
      f"{min(s['P_u']/(2*(s['P_u'] - p['P_C'])) for s in S):.1f} - {max(s['P_u']/(2*(s['P_u'] - p['P_C'])) for s in S):.1f}")

In [ ]:
# --- Verification of Correct Equations: Conservation of Mass ---
# To confirm our method converges to the *correct* equations (Criteria 3i),
# we verify that the total mass of nitrogen (m_p + m_u) is conserved.

m_p_ref = U_REF[:, 0]  # Pressurant tank N2 mass
m_u_ref = U_REF[:, 2]  # Ullage N2 mass
total_N2_mass = m_p_ref + m_u_ref

initial_N2_mass = u0[0] + u0[2]
mass_conservation_error = np.abs(total_N2_mass - initial_N2_mass) / initial_N2_mass

print(f"Initial Total N2 Mass: {initial_N2_mass:.6f} kg")
print(f"Max relative mass conservation error over the burn: {mass_conservation_error.max():.2e}")

fig, ax = plt.subplots(figsize=(7, 4))
# Add 1e-17 to avoid log(0) warnings if the error is exactly zero
ax.semilogy(T_REF, mass_conservation_error + 1e-17, color='indigo', lw=1.5)
ax.set(xlabel="t [s]", ylabel="Relative Error in Total N2 Mass",
       title="Verification of Equations: N2 Mass Conservation")
ax.grid(True, which="both", alpha=0.3)
fig.savefig(f"{FIG_DIR}/fig_conservation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from scipy.integrate import solve_ivp

# 1. Run SciPy's standard RK45 adaptive integrator
print("Running SciPy's RK45 integrator for comparison...")
sol_scipy = solve_ivp(
    fun=lambda t, u: f(u, p),     # solve_ivp expects signature f(t, y)
    t_span=(0, T_END),
    y0=u0,
    method='RK45',
    t_eval=T_REF,                 # Evaluate at the exact same times as the custom reference
    rtol=1e-8,
    atol=1e-8
)

# 2. Extract states and compute mass flow
u_scipy = sol_scipy.y.T
mf_scipy = np.array([mdot_f(u, p) for u in u_scipy])

# Calculate absolute difference between custom RK4 and SciPy RK45
abs_diff = np.abs(MF_REF - mf_scipy)

# 3. Plot the comparison and the residual difference
fig, axs = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Top subplot: Overlaid mass flow trajectories
axs[0].plot(T_REF, MF_REF, color='blue', linestyle='-', linewidth=2.5, label='Custom Method (RK4)')
axs[0].plot(sol_scipy.t, mf_scipy, color='orange', linestyle='--', linewidth=2, label="SciPy Stock Method (RK45)")
axs[0].set_ylabel(r"Fuel Mass Flow, $\dot{m}_f$ [kg/s]")
axs[0].set_title("Mass Flow Rate Comparison: Custom RK4 vs. SciPy RK45")
axs[0].grid(True, which="both", alpha=0.3)
axs[0].legend(loc="upper right")

# Bottom subplot: Absolute discrepancy
axs[1].semilogy(T_REF, abs_diff + 1e-18, color='red', linewidth=1.5, label="Absolute Difference")
axs[1].set_xlabel("Time, t [s]")
axs[1].set_ylabel(r"Difference [kg/s]")
axs[1].grid(True, which="both", alpha=0.3)
axs[1].legend(loc="upper right")

plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_scipy_comparison.pdf", bbox_inches="tight")
plt.show()

print(f"Maximum absolute discrepancy across the burn: {abs_diff.max():.3e} kg/s")

In [ ]:
# --- 6. Results: Meaningful System Performance (Report Sec 3) ---
# The rubric explicitly discourages plotting against time.
# To answer if the system maintains pressure until fuel is depleted,
# we plot against the percentage of fuel consumed.

# Calculate fuel consumed percentage from the reference RK4 burn (U_ref_b)
m_f0 = U_ref_b[0, 4]
fuel_consumed_pct = (m_f0 - U_ref_b[:, 4]) / m_f0 * 100

# Extract pressure and flow metrics from the state history
S_burn = [info(u, p) for u in U_ref_b]
P_u_psig = np.array([s["P_u"] for s in S_burn]) / PSI - 14.7
P_p_psig = np.array([s["P_p"] for s in S_burn]) / PSI - 14.7
mdot_f_vals = np.array([s["mdot_f"] for s in S_burn])

P_set_psig = p["P_SET"] / PSI - 14.7
P_c_psig = p["P_C"] / PSI - 14.7

# --- Create a Two-Panel Figure ---
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel 1: Ullage Pressure vs. Fuel Consumed
axs[0].plot(fuel_consumed_pct, P_u_psig, label=r'Ullage Pressure ($P_u$)', color='#1f77b4', lw=2.5)
axs[0].axhline(P_set_psig, color='k', linestyle='--', lw=1.5, label=r'Regulator Setpoint (360 psig)')
axs[0].axhline(P_c_psig, color='#d62728', linestyle='-.', lw=1.5, label=r'Engine Chamber Pressure ($P_e$, 300 psig)')
axs[0].fill_between(fuel_consumed_pct, P_c_psig, P_u_psig, color='#1f77b4', alpha=0.1)

axs[0].set_xlabel('Fuel Consumed [%]', fontsize=11)
axs[0].set_ylabel('Pressure [psig]', fontsize=11)
axs[0].set_title('Ullage Pressure Regulation Capability', fontsize=12, fontweight='bold')
axs[0].set_xlim(0, 100)
axs[0].legend(loc='upper right', fontsize=10)
axs[0].grid(True, which='major', linestyle='-', alpha=0.5)
axs[0].grid(True, which='minor', linestyle=':', alpha=0.2)
axs[0].minorticks_on()

# Panel 2: Pressurant Depletion and Engine Flow Rate vs. Fuel Consumed
ax2 = axs[1]
ax2_tw = ax2.twinx()

l1 = ax2.plot(fuel_consumed_pct, mdot_f_vals, color='#2ca02c', lw=2.5, label=r'Fuel Mass Flow ($\dot{m}_f$)')
l2 = ax2_tw.plot(fuel_consumed_pct, P_p_psig, color='#9467bd', lw=2.5, linestyle=':', label=r'Pressurant Tank Pressure ($P_p$)')

ax2.set_xlabel('Fuel Consumed [%]', fontsize=11)
ax2.set_ylabel(r'Fuel Mass Flow [kg/s]', color='#2ca02c', fontsize=11, fontweight='bold')
ax2_tw.set_ylabel(r'N$_2$ Tank Pressure [psig]', color='#9467bd', fontsize=11, fontweight='bold')

ax2.set_title('Engine Performance & Pressurant Depletion', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 100)

# Align the dual-axis legends
lines = l1 + l2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='upper right', fontsize=10)
ax2.grid(True, which='major', linestyle='-', alpha=0.5)
ax2.grid(True, which='minor', linestyle=':', alpha=0.2)
ax2.minorticks_on()

plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_results_meaningful.pdf", bbox_inches="tight")
plt.show()

print(f"Final N2 Tank Pressure at 100% fuel depletion: {P_p_psig[-1]:.0f} psig")
print(f"Minimum Ullage Pressure maintained: {P_u_psig.min():.1f} psig")